# Adding rsids to SNPs

A **genetic variant** is a specific location in the genome of a given species where the DNA sequence differs among individuals of that species. Each variant is defined by its precise position (**chromosome** and genomic **coordinate** on the forward strand), the consensus observed sequence at that position (**reference** allele), and one or more alternative sequences present in the population (**alternative** alleles).

Usually when we work with genetic variants, we tend to binarize them into reference allele vs alternative allele to facilitate comparisons across mutants. One way to identify genetic variants is in the "chromosome:position:reference:alternative" (cpra) format. Another, more popular, way to identify variants is by **r**eference **s**equence **id**entifier (**rsid**). 

By definition, cpra format identifiers are unique under the same variant calling pipeline. However since rsids only require position, at multi-allelic sites, they become non-identifiable. Due to this reason, most bioinformatics work proceed with alleles in cpra format and only add rsids to the variants during the final stages of the pipeline.

There are many techniques used to add rsids to genetic variants. This notebook explores 2 of them:
- avsnp (ANNOVAR)
- dbsnp (ncbi api)

# imports and globals

In [32]:
import pandas as pd
import requests
import re

import tabix
file="/nfs/corenfs/INTMED-Speliotes-data/External_Data/VEP/ncbi/GCF_000001405.40.gz"
if 'tb' not in locals() or not hasattr(tb, 'query'):
    tb = tabix.open(file)

from tqdm.notebook import tqdm

test_cpras = ["2:21002363:G:T", "2:21002393:A:T", "2:21002581:G:A", "2:21002683:G:A"]

# avsnp (using annovar)

Todo: add more background about avsnp from the annovar documentation (https://annovar.openbioinformatics.org/en/latest/articles/dbSNP/)

In [9]:
def get_rsids_avsnp(cpras,build="hg38",rm_temp=True):
    with open("av_rsid.input","w+") as f:
        for cpra in cpras:
            c,p,r,a = cpra.split(":")
            if c == "23":
                c = "X"
            p = int(p)
            if len(r) > 1 or len(a) > 1:
                line = f"{c}\t{p}\t{p+len(r)-1}\t{'0'}\t{a}\t{cpra}\n"
            else:
                line = f"{c}\t{p}\t{p+len(r)-1}\t{r}\t{a}\t{cpra}\n"
            f.write(line)
    ! /home/craut/wkspce/UKBB_500k_WES_15mill_variants/code/p3_make_gene_sets/annovar/annotate_variation.pl -filter -dbtype avsnp151 -buildver {build} -out av_rsid.ouput av_rsid.input /home/craut/wkspce/UKBB_500k_WES_15mill_variants/code/p3_make_gene_sets/annovar/humandb/ &> /dev/null
    annovar_file1 = "av_rsid.ouput.hg38_avsnp151_dropped"
    annovar_file2 = "av_rsid.ouput.hg38_avsnp151_filtered"
    in_db = pd.read_table(annovar_file1,header=None)
    in_db.columns=["db","rsid","CHROM","start","stop","REF","ALT","cpra"]
    not_in_db = pd.read_table(annovar_file2, header=None)
    not_in_db.columns=["CHROM","start","stop","REF","ALT","cpra"]
    cpra_2_rsid = {cpra:rsid for cpra,rsid in zip(in_db.cpra,in_db.rsid)}
    if rm_temp:
        ! rm -rf av_rsid.input
        ! rm -rf av_rsid.ouput.*
    return [cpra_2_rsid[cpra] if cpra in cpra_2_rsid else None for cpra in cpras]

In [10]:
# get rsids using avsnp
get_rsids_avsnp(test_cpras)

['rs935792706', None, None, 'rs907126709']

# dbsnp (using ncbi api)

Todo: add more background about the dbsnp records from the api documentation (https://api.ncbi.nlm.nih.gov/variation/v0/#/VCF/post_vcf_file_set_rsids)

In [25]:
def get_rsids_dbsnp(variant_strings, assembly="GCF_000001405.40", chunk_size=25_000, debug=False, consider_swaps=True):  
    if len(variant_strings) <= chunk_size:
        return _process_variant_chunk(variant_strings, assembly,debug, consider_swaps)
    
    all_rsids = []
    for i in range(0, len(variant_strings), chunk_size):
        chunk = variant_strings[i:i + chunk_size]
        chunk_rsids = _process_variant_chunk(chunk, assembly,debug, consider_swaps)
        all_rsids.extend(chunk_rsids)
    return all_rsids


def _process_variant_chunk(variant_strings, assembly, debug, consider_swaps):
    vcf_lines = ["##fileformat=VCFv4.2"]
    vcf_lines.append("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO")
    
    for variant in variant_strings:
        try:
            chrom, pos, ref, alt = variant.split(":")
            if not chrom.startswith("chr"):
                chrom = f"chr{chrom}"
            vcf_lines.append(f"{chrom}\t{pos}\t.\t{ref}\t{alt}\t.\t.\t.")
            if consider_swaps:
                vcf_lines.append(f"{chrom}\t{pos}\t.\t{alt}\t{ref}\t.\t.\t.")
        except ValueError:
            raise ValueError(f"Invalid variant format: {variant}. Expected 'chromosome:position:reference:alternative'")
    
    vcf_content = "\n".join(vcf_lines)
    
    url = f"https://api.ncbi.nlm.nih.gov/variation/v0/vcf/file/set_rsids?assembly={assembly}"
    headers = {
        'accept': 'text/plain; charset=utf-8',
        'Content-Type': 'text/plain; charset=utf-8'
    }
    
    try:
        response = requests.post(url, headers=headers, data=vcf_content)
        response.raise_for_status()
        rsids = []
        response_lines = response.text.strip().split('\n')
        for i,line in enumerate(response_lines):
            if line.startswith('#') or not line.strip():
                if debug:
                    print(line, response_lines[i+1])
                continue
            columns = line.split('\t')
            if len(columns) >= 3:
                rsid = columns[2]  # ID column
                if rsid and rsid != '.':
                    rsids.append(rsid)
                else:
                    rsids.append(None)
            else:
                rsids.append(None)
        if consider_swaps:
            resolved_swaps_rsid = []
            for a,b in zip(rsids[::2],rsids[1::2]):
                if pd.isna(a) and pd.isna(b):
                    resolved_swaps_rsid.append(None)
                elif pd.isna(a):
                    resolved_swaps_rsid.append(b)
                elif pd.isna(b):
                    resolved_swaps_rsid.append(a)
                else:
                    if debug:
                        print(f"conflicting rsids ({a} and {b}) for cpra: {variant_strings[len(resolved_swaps_rsid)]}")
                    resolved_swaps_rsid.append(None)
            rsids = resolved_swaps_rsid
        assert len(rsids) == len(variant_strings)
        return rsids
        
    except requests.exceptions.RequestException as e:
        raise Exception(f"API request failed: {e}")
    except Exception as e:
        raise Exception(f"Error processing response: {e}")

In [24]:
# get rsids using dbsnp
get_rsids_dbsnp(test_cpras)

['rs935792706', 'rs147825458', 'rs2465349460', 'rs907126709']

# dbsnp (using locally downloaded )

In [33]:
def chr_to_nc(chrom):
    nc_map = {
        1:  "NC_000001.11", 2:  "NC_000002.12", 3:  "NC_000003.12", 4:  "NC_000004.12",
        5:  "NC_000005.10", 6:  "NC_000006.12", 7:  "NC_000007.14", 8:  "NC_000008.11",
        9:  "NC_000009.12", 10: "NC_000010.11", 11: "NC_000011.10", 12: "NC_000012.12",
        13: "NC_000013.11", 14: "NC_000014.9",  15: "NC_000015.10", 16: "NC_000016.10",
        17: "NC_000017.11", 18: "NC_000018.10", 19: "NC_000019.10", 20: "NC_000020.11",
        21: "NC_000021.9",  22: "NC_000022.11", 23: "NC_000023.11", 24: "NC_000024.10",
        "X": "NC_000023.11", "Y": "NC_000024.10", "MT": "NC_012920.1"
    }
    if isinstance(chrom, int):
        return nc_map.get(chrom)
    chrom_str = str(chrom).upper()
    return nc_map.get(int(chrom_str) if chrom_str.isdigit() else chrom_str)

def local_ncbi_lookup(variant_strings,cpra_sep=":", debug=False, consider_swaps=True,return_swapped_bool=False):
    rsids = []
    swapped = []
    for v in tqdm(variant_strings):
        c,p,r,a = v.split(cpra_sep)
        c = c.split("chr")[-1]
        nc = chr_to_nc(c)
        p = int(p.split("[")[0])
        p0 = p-1
        p1 = p0+max(len(r),len(a))
        r = r.upper()
        a = a.upper()
        if debug:
            print(f"Processing variant {v} searching for record in ({nc}, {p0}, {p1})")
        records = [rec for rec in tb.query(nc, p0, p1)]
        if debug:
            print(f"Found {len(records)} records")
        for rec in records:
            chrom,pos,rsid,ref,alts,_,_,info = rec
            if int(pos) == p:
                alts = alts.upper().split(",")
                ref = ref.upper()
                if r == ref and a in alts:
                    rsids.append(rsid)
                    swapped.append(False)
                    if debug:
                        print(f"found perfect db match! {rsid}")
                    break
                elif consider_swaps and a == ref and r in alts:
                    rsids.append(rsid)
                    swapped.append(True)
                    if debug:
                        print(f"found swapped db match! {rsid} ({chrom}:{pos}:{ref}:{alts}, {info})")
                    break
        else:
            if debug:
                print("Could not find good db match :(")
            rsids.append(None)
            swapped.append(None)
    if return_swapped_bool:
        return rsids, swapped
    else:
        return rsids

In [34]:
local_ncbi_lookup(test_cpras)

  0%|          | 0/4 [00:00<?, ?it/s]

['rs935792706', 'rs147825458', 'rs2465349460', 'rs907126709']